[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dandi/example-notebooks/blob/master/tutorials/neuromatch_isp_2026/00_setup_check.ipynb)

## Installing requirements

The cell below installs every Python package needed to run this notebook, at fully pinned versions, using [`uv`](https://github.com/astral-sh/uv) for fast resolution. In Colab the cell is collapsed by default — click the ▶ button to run it.

In [ ]:
#@title Installing requirements (click ▶ to run) { display-mode: "form" }
# Colab provides Python 3.13. We install with `uv --system` because Colab's
# kernel runs outside a virtualenv. All versions (direct + transitive) are
# pinned below so the notebook is reproducible regardless of resolver drift.
!pip install -q uv
!uv pip install --system \
    "acres==0.5.0" \
    "aiohappyeyeballs==2.7.1" \
    "aiohttp==3.14.3" \
    "aiosignal==1.4.0" \
    "annotated-types==0.8.0" \
    "arrow==1.4.0" \
    "attrs==26.1.0" \
    "bids-validator-deno==3.0.2" \
    "bidsschematools==1.2.7" \
    "blessed==1.50.0" \
    "certifi==2026.7.22" \
    "cffi==2.1.1" \
    "charset-normalizer==3.4.9" \
    "ci-info==0.4.0" \
    "click==8.5.0" \
    "click-didyoumean==0.3.1" \
    "cloudpickle==3.1.2" \
    "contourpy==1.3.3" \
    "cryptography==50.0.1" \
    "cycler==0.12.1" \
    "dandi==0.82.0" \
    "dandischema==0.14.0" \
    "deno==2.9.8" \
    "dnspython==2.9.2" \
    "donfig==0.8.1.post1" \
    "email-validator==2.3.0" \
    "etelemetry==0.3.1" \
    "fasteners==0.20" \
    "fonttools==4.64.0" \
    "fqdn==1.5.1" \
    "frozenlist==1.8.0" \
    "fscacher==0.5.0" \
    "fsspec==2025.12.0" \
    "google-crc32c==1.8.0" \
    "h5py==3.16.0" \
    "hdmf==4.3.1" \
    "humanize==4.16.0" \
    "idna==3.19" \
    "interleave==0.3.0" \
    "isodate==0.7.2" \
    "isoduration==20.11.0" \
    "jaraco-classes==3.4.0" \
    "jaraco-context==6.1.2" \
    "jaraco-functools==4.6.0" \
    "jeepney==0.9.0" \
    "jinxed==2.1.0" \
    "joblib==1.6.0" \
    "jsonpointer==3.1.1" \
    "jsonschema==4.26.0" \
    "jsonschema-specifications==2025.9.1" \
    "keyring==25.7.0" \
    "keyrings-alt==5.0.2" \
    "kiwisolver==1.5.1" \
    "matplotlib==3.10.0" \
    "ml-dtypes==0.6.0" \
    "more-itertools==10.8.0" \
    "msgspec==0.22.0" \
    "multidict==6.7.1" \
    "natsort==8.4.0" \
    "numcodecs==0.17.1" \
    "numpy==2.1.3" \
    "nwbinspector==0.7.2" \
    "packaging==26.3" \
    "pandas==2.2.3" \
    "pillow==11.3.0" \
    "platformdirs==4.11.7" \
    "propcache==0.5.2" \
    "pycparser==3.0" \
    "pycryptodomex==3.23.0" \
    "pydantic==2.13.5" \
    "pydantic-core==2.46.5" \
    "pydantic-settings==2.16.2" \
    "pynwb==3.1.3" \
    "pyout==0.8.1" \
    "pyparsing==3.3.2" \
    "python-dateutil==2.9.0.post0" \
    "python-dotenv==1.2.3" \
    "pytz==2025.2" \
    "pyyaml==6.0.3" \
    "referencing==0.37.0" \
    "remfile==0.1.15" \
    "requests==2.32.4" \
    "rfc3339-validator==0.1.4" \
    "rfc3987==1.3.8" \
    "rpds-py==2026.6.3" \
    "ruamel-yaml==0.19.1" \
    "secretstorage==3.5.0" \
    "semantic-version==2.10.0" \
    "six==1.17.0" \
    "tenacity==9.1.4" \
    "tensorstore==0.1.85" \
    "tqdm==4.67.3" \
    "typing-extensions==4.16.0" \
    "typing-inspection==0.4.4" \
    "tzdata==2026.3" \
    "uri-template==1.3.0" \
    "urllib3==2.5.0" \
    "wcwidth==0.8.3" \
    "webcolors==25.10.0" \
    "yarl==1.24.5" \
    "zarr==3.4.2" \
    "zarr-checksum==0.4.7"

> **⚠️ Restart runtime after install**
>
> The install may upgrade packages already loaded in the kernel. Go to **Runtime → Restart session**, then **Run all cells below** (skip this install cell on re-run).

# Setup check: DANDI + NWB workshop

Neuromatch Impact Scholars Program, 30 November 2026.

Please run this notebook once **before** the workshop. It takes about 5 minutes.

1. Run the install cell above.
2. Restart the runtime when asked (**Runtime → Restart session**).
3. Run the cells below.

If the last cell prints ✅, you are ready.

In [1]:
import time
import warnings

import h5py
import remfile
from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO

warnings.filterwarnings("ignore")

The next cell opens one recording session from the International Brain Laboratory on the DANDI Archive.

The file is streamed. Only the parts we ask for travel over the network.

In [2]:
start = time.time()

client = DandiAPIClient()
dandiset = client.get_dandiset("000409", "0.260309.1324")
asset = dandiset.get_asset_by_path(
    "sub-DY-011/sub-DY-011_ses-cf43dbb1-6992-40ec-a5f9-e8e838d0f643_desc-processed_behavior+ecephys.nwb"
)

url = asset.get_content_url(follow_redirects=1, strip_query=True)
nwbfile = NWBHDF5IO(file=h5py.File(remfile.File(url), mode="r"), load_namespaces=True).read()
trials = nwbfile.trials.to_dataframe()

elapsed = time.time() - start
print(f"File size on DANDI: {asset.size / 1e6:.0f} MB")
print(f"Opened and read {len(trials)} trials in {elapsed:.0f} seconds")
print("✅ You are ready for the workshop!")

File size on DANDI: 345 MB
Opened and read 402 trials in 7 seconds
✅ You are ready for the workshop!


### On the day

Colab does not keep installed packages between sessions.
At the start of the workshop you will run the install cell of the workshop notebook again. It takes about a minute.

Something did not work? Tell the organizers which cell failed and paste the error message.